In [1]:
import re
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.tools import tool

In [2]:
df_chunks = pd.read_json("rag_chunks_v1.jsonl", lines=True)
meta_df = pd.json_normalize(df_chunks["metadata"])
df_rag = pd.concat([df_chunks[["chunk_id", "text"]], meta_df], axis=1)

In [3]:
def tokenize_vi(text: str):
    return re.sub(r"[^\w\s]", " ", text.lower()).split()

bm25_index = BM25Okapi([tokenize_vi(doc) for doc in df_rag["text"]])
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=15000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df_rag["text"])

In [4]:
@tool
def search_parking_policy(query: str, city: str = "ALL") -> str:
    """
    Tra cứu quy định, nội quy ban quản lý, bảng giá gửi xe, quy chuẩn kỹ thuật
    hoặc quy trình xử lý sự cố (mất thẻ, lỗi camera, đỗ sai chỗ) từ kho tài liệu bãi đỗ xe.
    - query: Câu hỏi hoặc từ khóa cần tìm.
    - city: Khu vực ('HN', 'HCM', hoặc 'ALL').
    """
    mask = df_rag["city"].isin([city, "ALL"]) if city != "ALL" else pd.Series([True] * len(df_rag))
    valid_indices = df_rag[mask].index.tolist()

    bm25_scores = np.array(bm25_index.get_scores(tokenize_vi(query)))
    if bm25_scores.max() > 0:
        bm25_scores = bm25_scores / bm25_scores.max()

    cos_scores = cosine_similarity(tfidf_vectorizer.transform([query]), tfidf_matrix).flatten()
    final_scores = 0.55 * bm25_scores + 0.45 * cos_scores

    top_indices = sorted(valid_indices, key=lambda idx: final_scores[idx], reverse=True)[:3]
    
    results = []
    for idx in top_indices:
        row = df_rag.loc[idx]
        results.append(f"[Nguồn: {row['source_file']}]\n{row['text']}")
    return "\n\n---\n\n".join(results)

In [5]:
# Dữ liệu giả lập (Mock Data)
MOCK_SLOTS = [
    {"slot_id": "B1-A05", "type": "TH1", "car_size": "SUV", "time": "08:00-17:00", "price": 50000, "status": "available"},
    {"slot_id": "B1-B12", "type": "TH2", "car_size": "Sedan", "time": "08:00-18:00", "price": 45000, "status": "available"},
    {"slot_id": "B2-C01", "type": "TH3", "car_size": "SUV", "time": "24/7", "price": 60000, "status": "available"},
]

@tool
def check_parking_eligibility(user_role: str, requested_type: str) -> dict:
    """
    Kiểm tra quyền được phép thuê loại chỗ đỗ (Rule Engine mô phỏng).
    - user_role: Vai trò người dùng ('resident' = cư dân, 'shophouse_owner' = chủ shophouse, 'visitor' = khách vãng lai/khách shophouse).
    - requested_type: Loại chỗ muốn thuê ('TH1', 'TH2', 'TH3').
    """
    # Quy tắc nghiệp vụ: Khách vãng lai chỉ được đỗ TH3[cite: 3]
    if user_role == "visitor" and requested_type in ["TH1", "TH2"]:
        return {
            "allowed": False,
            "redirect_to": "TH3",
            "reason": "Khách vãng lai / khách shophouse không được thuê chỗ riêng TH1/TH2, chỉ được đỗ khu công cộng TH3[cite: 3]."
        }
    return {"allowed": True, "assigned_type": requested_type, "reason": "Đủ điều kiện hợp lệ."}

@tool
def find_available_slots(slot_type: str, car_size: str) -> list:
    """
    Tìm danh sách các ô đỗ đang trống theo loại chỗ và kích thước xe.
    - slot_type: 'TH1', 'TH2', hoặc 'TH3'.
    - car_size: 'Sedan' hoặc 'SUV'.
    """
    matched = [
        s for s in MOCK_SLOTS 
        if s["type"] == slot_type and s["status"] == "available"
        and (s["car_size"] == car_size or s["car_size"] == "SUV") # Ô SUV chứa được cả Sedan
    ]
    return matched if matched else [{"message": f"Hiện hết chỗ trống loại {slot_type} cho xe {car_size}."}]

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

os.environ["GOOGLE_API_KEY"] = ""  # Thay API Key của bạn vào đây

# 1. Khởi tạo LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.1)

In [9]:


# 2. Gom tất cả tools vào 1 danh sách
tools = [search_parking_policy, check_parking_eligibility, find_available_slots]

# 3. Viết System Prompt định hướng hành vi cho Agent
system_prompt = """
Bạn là Trợ lý AI Điều phối Bãi đỗ xe Thông minh.
Nguyên tắc hoạt động bắt buộc:
1. Khi người dùng hỏi về quy định, bảng giá, luật, hoặc cách xử lý sự cố (mất thẻ, lỗi camera...): LUÔN gọi tool `search_parking_policy` để lấy thông tin chính xác và trích dẫn nguồn.
2. Khi người dùng muốn tìm/thuê chỗ đỗ xe:
   - Bước 1: Gọi `check_parking_eligibility` để kiểm tra quyền của họ trước.
   - Bước 2: Nếu bị từ chối TH1/TH2, giải thích lịch sự và tự động tìm chỗ ở loại được chuyển hướng (TH3) bằng `find_available_slots`.
   - Bước 3: Chỉ gợi ý các mã ô đỗ có trong kết quả trả về từ tool, tuyệt đối không tự bịa mã ô đỗ.
"""

# 4. Tạo Agent (LangGraph tự động dựng luồng Thought -> Tool Call -> Observation -> Answer)
parking_agent = create_react_agent(model=llm, tools=tools, prompt=system_prompt)

C:\Users\leewi\AppData\Local\Temp\ipykernel_11568\725801325.py:16: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  parking_agent = create_react_agent(model=llm, tools=tools, prompt=system_prompt)


In [10]:
def chat_with_agent(user_message: str):
    print(f"\n👤 USER: {user_message}")
    print("-" * 60)
    
    # Gọi Agent thực thi
    result = parking_agent.invoke({"messages": [("user", user_message)]})
    
    # In ra quá trình Agent gọi tool (để bạn dễ debug)
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"🛠️  [Agent gọi Tool]: {tc['name']} | Tham số: {tc['args']}")
                
    # In câu trả lời cuối cùng
    final_answer = result["messages"][-1].content
    print(f"\n🤖 AGENT:\n{final_answer}")
    print("=" * 60)

# Kịch bản 1: Chỉ gọi RAG Tool
chat_with_agent("Khách làm mất thẻ từ gửi xe thì quy trình xử lý thế nào?")

# Kịch bản 2: Gọi chuỗi Mock Tools (Rule Engine -> Tìm chỗ trống)
chat_with_agent("Tôi là khách vãng lai (visitor) đi xe SUV, muốn thuê chỗ riêng TH1 từ 8h đến 17h.")

# Kịch bản 3: Kết hợp cả Rule Engine + Tìm chỗ + RAG tra cứu giá/quy định
chat_with_agent("Tôi là cư dân (resident) đi xe Sedan muốn tìm chỗ TH2, tiện thể cho hỏi giá gửi xe ô tô dưới 9 chỗ theo tháng ở Hà Nội là bao nhiêu?")


👤 USER: Khách làm mất thẻ từ gửi xe thì quy trình xử lý thế nào?
------------------------------------------------------------
🛠️  [Agent gọi Tool]: search_parking_policy | Tham số: {'city': 'ALL', 'query': 'mất thẻ từ quy trình xử lý'}
🛠️  [Agent gọi Tool]: search_parking_policy | Tham số: {'city': 'ALL', 'query': 'mất thẻ xe quy trình xử lý mất thẻ'}

🤖 AGENT:
[{'type': 'text', 'text': 'Theo quy trình xử lý sự cố của hệ thống bãi đỗ xe (*Nguồn: Cẩm nang Xử lý Sự cố & Kịch bản CSKH Bãi đỗ xe.pdf* và *Bai_Do_Xe_Thong_Minh_Su_Dung_The_Tu.docx*), việc báo mất thẻ **không được coi là căn cứ duy nhất để cho xe ra hay mở barrier ngay lập tức**. \n\nNhân viên vận hành và Ban Quản lý sẽ triển khai theo **quy trình 4 bước** như sau:\n\n---\n\n### **Quy trình 4 bước xử lý khi khách làm mất thẻ từ:**\n\n1. **Bước 1: Hướng dẫn xe vào vị trí chờ**\n   * Nếu làn ra đang đông, nhân viên hướng dẫn khách di chuyển xe vào làn xử lý sự cố hoặc vị trí sát lề an toàn để tránh gây ùn tắc giao thông.\n\n2. 